In [ ]:
import requests
import json

# Download the data
resp = requests.get('https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json')
data = json.loads(resp.text)  # Load data
    
def json_print(data):
    print(json.dumps(data, indent=2))
    
json_print(data)

In [ ]:
import weaviate
import os

client = weaviate.connect_to_embedded(
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]  # Replace this with your actual key
    }
)

In [ ]:
client.collections.delete("Question")

In [ ]:
from weaviate.classes.config import Configure

client.collections.create(
    name="Question",
    vectorizer_config=Configure.Vectorizer.TEXT2VEC_OPENAI
)

In [ ]:
questions = client.collections.get("Question")

# Batch import data
with questions.batch.dynamic() as batch:
    for i, d in enumerate(data):
        
        print(f"importing question: {i+1}")
        
        properties = {
            "answer": d["Answer"],
            "question": d["Question"],
            "category": d["Category"],
        }
        
        batch.add_object(
            properties=properties,
        )

In [ ]:
questions = client.collections.get("Question")
response = questions.aggregate.over_all(total_count=True)

print(response.total_count)

### Lets run a vector search to see whats comes back

In [ ]:
#Write a vector search related to animals

response = questions.query.near_text(
    query="animals in movies",
    limit=2
)

for o in response.objects:
    json_print(o.properties)

### No we want to pass each of these objects to a LLM individually to use when answering a prompt!

In [ ]:
#Write a prompt that will be passed in the returend object above.

# ADD CODE HERE

In [ ]:
#Write a query to perform RAG

response = questions.generate.near_text(
    query="animals in movies",
    limit=2,
    single_prompt="Write a short story using these facts: {question}",
)

for o in response.objects:
    json_print(o.properties)
    print(o.generated)

### Lets extract all the categories

In [ ]:
response = questions.query.near_text(
    query="animals in movies",
    limit=10,
    return_properties=["category"]
)

for o in response.objects:
    json_print(o.properties)

### Now we'll pass all of these in at the same time for a LLM to generate a grouped answer.

In [ ]:
#Write a prompt that requires information from all returned objects

# ADD CODE HERE

In [ ]:
#write a query that generates a grouped response

response = questions.generate.near_text(
    query="animals in movies",
    limit=2,
    grouped_task="Write a short story using these facts.",
)

print(response.generated)